# Governance Dashboard — QuickSight Analysis for Inference & Drift Monitoring

> 📊 **Best viewed on [nbviewer](https://nbviewer.org/github/aws-samples/sample-mlops-bestpractices/blob/main/sagemaker-automated-drift-and-trend-monitoring/notebooks/4_governance_dashboard.ipynb)** — GitHub's renderer strips JavaScript that powers interactive output cells (Evidently reports, plotly charts, ipywidgets). nbviewer (run by Project Jupyter) renders them in full.


This notebook programmatically creates a complete QuickSight dashboard — no manual UI steps.

**Schema awareness:** the feature-drift dataset (Sheet 3) and the inference dataset both expose `monitoring_run_id`. The drift Lambda back-fills this column on `inference_responses` for every row it scored, so the cross-dataset JOIN is now an exact foreign key (was a 24-hour time-window approximation in earlier versions).

**Visuals — Sheet 1 (Inference Monitoring):**
1. Prediction Volume Over Time
2. Fraud Probability Distribution
3. Prediction Accuracy Breakdown
4. Risk Tier Distribution
5. Inference Latency Trend
6. Total Inferences KPI
7. Model Prediction Accuracy Over Time (%) — NEW
8. Daily Predictions: Correct vs Incorrect — NEW

**Visuals — Sheet 2 (Drift Trend Analysis):**
7. Data Drift Share Over Time (line chart)
8. Drifted Columns Count Trend (bar chart)
9. Model Drift Detection (ROC-AUC comparison)
10. Data Drift Score Distribution (histogram)
11. Monitoring Run Volume (bar chart)
12. Drift Detection KPI (latest drift share)

**Visuals — Sheet 3 (Feature Drift Analysis by Model Version):**
13. Drift by Model Version Over Time (multi-line showing drift evolution per version)
14. Model Version Performance Summary (table with drift, ROC-AUC, run counts)
15. Inference Volume by Model Version (stacked area chart)
16. Drift Intensity Heatmap (pivot table: time × version)

**Visuals — Sheet 4 (Feature-Level Drift Detail):**
17. Top Drifted Features (bar chart)
18. Feature Drift Score Trend (line chart with filter)
19. Feature Drift Details (table)
20. Feature Stability Overview (KPI card)
21. Feature Drift Distribution (histogram)
22. Feature Drift Heatmap (pivot: features × time)


## What this notebook does

This notebook builds (or refreshes) the QuickSight **governance dashboard** — the
inference, drift, feature-drift, and prediction-accuracy visuals — on top of the
Athena tables written by the drift-monitoring pipeline.

**All of the logic lives in one place:** `src/governance/create_governance_dashboard.py`.
That module is the single source of truth for every dataset, Athena view, visual, analysis,
and dashboard definition. It is also what `main.py dashboard create` calls. This notebook is
a thin driver over that module — it does **not** redefine any datasets or visuals inline, so
the notebook and the CLI can never drift out of sync.

> Because everything is defined in the module, the feature-drift visuals automatically use the
> test-agnostic **`drift_magnitude`** field (higher = more drift, regardless of which statistical
> test Evidently picked) instead of the ambiguous raw `drift_score`.

## 1. Setup & Configuration

In [ ]:
import sys, boto3
from pathlib import Path
from dotenv import load_dotenv

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
load_dotenv(project_root / '.env')

from src.config.config import AWS_DEFAULT_REGION, QUICKSIGHT_IDENTITY_REGION, ATHENA_DATABASE

# Everything the notebook does is delegated to this module — the single
# source of truth for datasets, the feature_drift_detail view, visuals,
# the analysis, and the dashboard. No dataset/visual is redefined here.
from src.governance import create_governance_dashboard as gov

# Account ID for the QuickSight/Athena calls below.
sts = boto3.client('sts', region_name=AWS_DEFAULT_REGION)
ACCOUNT_ID = sts.get_caller_identity()['Account']

print(f'Account:                    {ACCOUNT_ID}')
print(f'Data-plane region (Athena): {AWS_DEFAULT_REGION}')
print(f'QuickSight identity region: {QUICKSIGHT_IDENTITY_REGION}')
print(f'Database:                   {ATHENA_DATABASE}')

> ⚠️ **First-time QuickSight setup required (once per AWS account).**
>
> If this is a brand-new account, `describe_account_settings` in the next cell will fail with `ResourceNotFoundException` or similar. QuickSight has to be signed up for in the AWS console first — the CLI/notebook can't do it for you. **5-minute one-time setup:**
>
> 1. AWS console → search **QuickSight** → **Sign up for QuickSight**.
> 2. Edition: **Enterprise** (Standard doesn't support the Definition API this notebook uses).
> 3. Auth: **Use IAM federated identities and QuickSight-managed users** (default).
> 4. Region: pick your **identity region** — QuickSight fixes this per account at sign-up. Usually `us-east-1`. If yours is different, set `QUICKSIGHT_IDENTITY_REGION=<region>` in `.env` before running this notebook.
> 5. QuickSight console → username (top right) → **Manage QuickSight** → **Manage users** → **Invite users** — add your IAM/SSO identity as **Author** or **Admin**.
> 6. Same page → **Security & permissions** → **Manage QuickSight access to AWS services** → check **Amazon S3** (select the base stack's data bucket) and **Amazon Athena**. Save.
>
> Verify by visiting [https://quicksight.aws.amazon.com/](https://quicksight.aws.amazon.com/) — you should see the QuickSight home page. Then rerun the cell below.
>
> Full prerequisites + troubleshooting: see the README's **QuickSight prerequisites (one-time per account)** section.

## 2. Build the dashboard (one shot)

`gov.create_dashboard()` runs the entire flow end-to-end, in order:

1. Resolve account → check QuickSight subscription → verify Athena has data
2. Resolve QuickSight principals → create/update the Athena **data source**
3. Grant Lake Formation + S3 permissions
4. Create/update the **datasets**: inference, drift (`monitoring_responses`),
   feature-drift join, feature-level (from the `feature_drift_detail` view), accuracy
5. Create/replace the **`feature_drift_detail` Athena view** (+ its Lake Formation grant)
6. Create/update the **analysis**, then publish the **dashboard**
7. Best-effort **embed URL**

This is idempotent — re-run it any time to pick up code changes in the module.

> The visuals are all `DIRECT_QUERY`, so they run their SQL live against Athena at
> render time. Running this cell republishes the definitions; open the dashboard in
> QuickSight afterward to see the refreshed visuals.

In [ ]:
result = gov.create_dashboard()

print()
print('=' * 70)
print('Dashboard build complete')
print('=' * 70)
print(f"QuickSight subscribed: {result['quicksight_subscribed']} (edition: {result['quicksight_edition']})")
print(f"Dashboard URL:  {result['dashboard_url']}")
print(f"Dashboard ARN:  {result['dashboard_arn']}")
print(f"Analysis ARN:   {result['analysis_arn']}")
print()
print('Datasets:')
for k in ('inference', 'drift', 'feature_drift', 'feature_level', 'accuracy'):
    print(f"  {k:14s} {result[f'{k}_dataset_arn']}")
if result.get('embed_url'):
    print(f"\nEmbed URL (valid 10h):\n  {result['embed_url']}")

## 3. Step-by-step (optional / for debugging)

The cells below reproduce `create_dashboard()` one section at a time by calling the
**same module functions** it uses internally. Use these when you want to re-run a
single stage (e.g. just recreate the Athena view, or just republish the dashboard)
without redoing everything. Skip this whole section if you already ran section 2.

Run the setup cell first to build the shared clients and resolve the QuickSight principals.

In [ ]:
# Shared clients + principals used by the step-by-step calls below.
quicksight_admin = boto3.client('quicksight', region_name=QUICKSIGHT_IDENTITY_REGION)
quicksight       = boto3.client('quicksight', region_name=AWS_DEFAULT_REGION)
athena           = boto3.client('athena',     region_name=AWS_DEFAULT_REGION)

# Verify QuickSight is active + Athena has data (both log warnings, don't raise).
gov.check_quicksight_subscription(account_id=ACCOUNT_ID, quicksight_admin_client=quicksight_admin)
gov.verify_athena_data(athena_client=athena)

QS_PRINCIPALS = gov.get_quicksight_principals(account_id=ACCOUNT_ID, quicksight_admin_client=quicksight_admin)
print(f'QuickSight principals: {QS_PRINCIPALS}')

### 3a. Data source + permissions

In [ ]:
DATASOURCE_ARN = gov.create_or_update_datasource(
    account_id=ACCOUNT_ID, quicksight_principals=QS_PRINCIPALS, quicksight_client=quicksight,
)
gov.grant_governance_permissions(region=AWS_DEFAULT_REGION)
print(f'Data source: {DATASOURCE_ARN}')

### 3b. Datasets + feature_drift_detail view

Order matters: the `feature_drift_detail` view must exist before the feature-level
dataset references it. The view parses the per-feature JSON
(`{score, magnitude, method, threshold}`) and exposes `drift_magnitude` / `drift_method`.

In [ ]:
inference_arn     = gov.create_inference_dataset(DATASOURCE_ARN, ACCOUNT_ID, QS_PRINCIPALS, quicksight_client=quicksight)
drift_arn         = gov.create_drift_dataset(DATASOURCE_ARN, ACCOUNT_ID, QS_PRINCIPALS, quicksight_client=quicksight)
feature_drift_arn = gov.create_feature_drift_dataset(DATASOURCE_ARN, ACCOUNT_ID, QS_PRINCIPALS, quicksight_client=quicksight)

# Create/replace the Athena view, grant it, then the dataset that reads it.
gov.create_feature_drift_detail_view(athena_client=athena)
gov.grant_feature_drift_view_permissions(region=AWS_DEFAULT_REGION)
feature_level_arn = gov.create_feature_level_dataset(DATASOURCE_ARN, ACCOUNT_ID, QS_PRINCIPALS, quicksight_client=quicksight)

accuracy_arn = gov.create_accuracy_dataset(DATASOURCE_ARN, ACCOUNT_ID, QS_PRINCIPALS, quicksight_client=quicksight)

DATASET_ARNS = {
    'inference': inference_arn,
    'drift': drift_arn,
    'feature_drift': feature_drift_arn,
    'feature_level': feature_level_arn,
    'accuracy': accuracy_arn,
}
DATASET_ARNS

### 3c. Analysis + dashboard

The visual definitions (Model Drift, Data Drift, and Feature Drift sheets — the last
built by `build_feature_drift_visuals()` using `drift_magnitude`) are assembled inside
these calls from the module. Nothing is defined in the notebook.

In [ ]:
analysis_arn = gov.create_or_update_analysis(
    ACCOUNT_ID, QS_PRINCIPALS, DATASET_ARNS, quicksight_client=quicksight,
)
dashboard_result = gov.publish_dashboard(
    ACCOUNT_ID, QS_PRINCIPALS, DATASET_ARNS, quicksight_client=quicksight, region=AWS_DEFAULT_REGION,
)
print(f"Analysis ARN:  {analysis_arn}")
print(f"Dashboard URL: {dashboard_result['dashboard_url']}")

## 4. Generate Embed URL (optional)

In [ ]:
embed_url = gov.get_dashboard_embed_url(account_id=ACCOUNT_ID)
if embed_url:
    print(f'Embed URL (valid 10 hours):\n  {embed_url}')
    print(f'\nHTML:\n  <iframe src="{embed_url}" width="100%" height="800px"></iframe>')
else:
    print('Embed URL not available — ensure embedding is enabled in QuickSight admin settings.')

## 5. Cleanup (optional)

Deletes the dashboard, analysis, all datasets, and the data source via
`gov.delete_dashboard()`. Guarded by a flag so you don't wipe the dashboard by accident.

In [ ]:
CONFIRM_DELETE = False  # set True to delete all governance QuickSight resources

if CONFIRM_DELETE:
    outcome = gov.delete_dashboard(region=AWS_DEFAULT_REGION)
    print('Deleted:  ', outcome['deleted'])
    print('Not found:', outcome['not_found'])
    print('Errors:   ', outcome['errors'])
else:
    print('Cleanup skipped — set CONFIRM_DELETE = True to run.')